In [ ]:
class MultiTaskBERT(nn.Module):

    def __init__(self):

        super().__init__()

        self.bert = AutoModel.from_pretrained("bert-base-cased")

        hidden = self.bert.config.hidden_size

        self.dropout = nn.Dropout(0.1)

        self.punct_classifier = nn.Linear(hidden, 4)
        self.cap_classifier   = nn.Linear(hidden, 3)

        self.punct_loss_fct = nn.CrossEntropyLoss(
            weight=torch.tensor([0.1,1.5,1.0,2.0], dtype=torch.float),
            ignore_index=-100
        )

        self.cap_loss_fct = nn.CrossEntropyLoss(
            weight=torch.tensor([0.1,1.0,1.2], dtype=torch.float),
            ignore_index=-100
        )
    def forward(self, input_ids, attention_mask,
                punct_labels=None, cap_labels=None):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden_states = outputs.last_hidden_state
        hidden_states = self.dropout(hidden_states)

        punct_logits = self.punct_classifier(hidden_states)
        cap_logits   = self.cap_classifier(hidden_states)

        loss = None

        if punct_labels is not None and cap_labels is not None:

            punct_loss = self.punct_loss_fct(
                punct_logits.view(-1,4),
                punct_labels.view(-1)
            )

            cap_loss = self.cap_loss_fct(
                cap_logits.view(-1,3),
                cap_labels.view(-1)
            )

            loss = punct_loss + cap_loss

        return {
            "loss": loss,
            "punct_logits": punct_logits,
            "cap_logits": cap_logits
        }

model = MultiTaskBERT()

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1
)

model.bert = get_peft_model(model.bert, config)

model = model.to(device)

model.bert.print_trainable_parameters()